# Augmentation Visualizer

Visualises each candidate appearance augmentation on a synthetic 224x224 grayscale image.
No TensorFlow/Keras required -- pure NumPy + PIL.

Categories: **Tone | Noise | Random Lines | Edge Effects | Blur | Random Profiles**

---

## Table of Contents

| # | Category | Augmentations |
|---|---|---|
| 1 | [Tone](#cat-tone) | Brightness shift, Contrast, Gamma, Exposure simulation, Fog/haze |
| 2 | [Noise](#cat-noise) | Gaussian noise, Salt-and-pepper |
| 3 | [Random Lines](#cat-lines) | Horizontal lines, Vertical lines |
| 4 | [Edge Effects](#cat-edge) | Horizontal fading, Vertical fading, Horizontal bilateral, Vertical bilateral |
| 5 | [Blur](#cat-blur) | Gaussian blur |
| 6 | [Random Profiles](#cat-random) | Horizontal profile, Vertical profile |
| -- | [Combination](#cat-combo) | Chained augmentations |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter

## Synthetic image

Noisy mid-grey background with three solid rectangles of different tones.

In [ ]:
def make_synthetic(seed: int = 42) -> np.ndarray:
    """Return a 224x224 uint8 grayscale array."""
    rng = np.random.default_rng(seed)
    img = rng.integers(110, 145, (224, 224), dtype=np.uint8)
    rects = [
        (20,  30,  110, 120, 200),
        (130, 50,  210, 150,  50),
        (60,  140, 170, 200, 140),
    ]
    for x1, y1, x2, y2, shade in rects:
        img[y1:y2, x1:x2] = shade
    return img

ORIGINAL = make_synthetic()

plt.figure(figsize=(3, 3))
plt.imshow(ORIGINAL, cmap='gray', vmin=0, vmax=255)
plt.title('Synthetic original')
plt.axis('off')
plt.tight_layout()
plt.show()

## Helper: side-by-side comparison

In [ ]:
def show_variants(original: np.ndarray, variants: list, title: str):
    """Show original + variants in a single row. variants: list of (label, ndarray)."""
    n = 1 + len(variants)
    fig, axes = plt.subplots(1, n, figsize=(n * 3, 3))
    axes[0].imshow(original, cmap='gray', vmin=0, vmax=255)
    axes[0].set_title('original')
    axes[0].axis('off')
    for ax, (label, img) in zip(axes[1:], variants):
        ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        ax.set_title(label)
        ax.axis('off')
    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

def show_grid(fn, param_name: str, param_values: list, seeds: list, title: str):
    """Show a grid: rows=param values, cols=seeds."""
    rows, cols = len(param_values), len(seeds)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    if rows == 1:
        axes = [axes]
    for r, val in enumerate(param_values):
        for c, seed in enumerate(seeds):
            axes[r][c].imshow(fn(ORIGINAL, **{param_name: val}, seed=seed),
                              cmap='gray', vmin=0, vmax=255)
            axes[r][c].set_title(f'{param_name}={val}  seed={seed}', fontsize=8)
            axes[r][c].axis('off')
    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

<a id="cat-tone"></a>
# Category 1: Tone

Augmentations that shift overall pixel intensity - brightness, contrast curves, and haze.

---
## 1.1 Brightness shift

Uniformly add or subtract a constant from every pixel.

In [ ]:
def aug_brightness(img: np.ndarray, delta: int) -> np.ndarray:
    """Shift pixel values by delta (+bright, -dark). Clips to [0, 255]."""
    return np.clip(img.astype(np.int16) + delta, 0, 255).astype(np.uint8)

show_variants(ORIGINAL, [
    ('delta = -60', aug_brightness(ORIGINAL, -60)),
    ('delta = -30', aug_brightness(ORIGINAL, -30)),
    ('delta = +30', aug_brightness(ORIGINAL, +30)),
    ('delta = +60', aug_brightness(ORIGINAL, +60)),
], 'Brightness shift')

---
## 1.2 Contrast adjustment

Scale pixel distances from the mid-point (128). `factor > 1` increases contrast, `0 < factor < 1` reduces it.

In [ ]:
def aug_contrast(img: np.ndarray, factor: float) -> np.ndarray:
    """Multiply contrast around mid-grey (128) by factor."""
    out = 128 + factor * (img.astype(np.float32) - 128)
    return np.clip(out, 0, 255).astype(np.uint8)

show_variants(ORIGINAL, [
    ('factor = 0.3', aug_contrast(ORIGINAL, 0.3)),
    ('factor = 0.7', aug_contrast(ORIGINAL, 0.7)),
    ('factor = 1.5', aug_contrast(ORIGINAL, 1.5)),
    ('factor = 2.0', aug_contrast(ORIGINAL, 2.0)),
], 'Contrast adjustment')

---
## 1.3 Gamma correction

Non-linear tone curve: out = 255 * (in/255)^gamma.
gamma < 1 brightens shadows; gamma > 1 darkens them.

In [ ]:
def aug_gamma(img: np.ndarray, gamma: float) -> np.ndarray:
    """Apply gamma correction. gamma<1 brightens, gamma>1 darkens."""
    out = 255.0 * (img.astype(np.float32) / 255.0) ** gamma
    return np.clip(out, 0, 255).astype(np.uint8)

show_variants(ORIGINAL, [
    ('gamma = 0.3', aug_gamma(ORIGINAL, 0.3)),
    ('gamma = 0.7', aug_gamma(ORIGINAL, 0.7)),
    ('gamma = 1.5', aug_gamma(ORIGINAL, 1.5)),
    ('gamma = 2.5', aug_gamma(ORIGINAL, 2.5)),
], 'Gamma correction')

---
## 1.4 Exposure simulation

Multiply pixel values by 2^stops - mimics opening/closing a camera aperture by N f-stops.

In [ ]:
def aug_exposure(img: np.ndarray, stops: float) -> np.ndarray:
    """Simulate exposure change by `stops` f-stops (positive=brighter)."""
    out = img.astype(np.float32) * (2.0 ** stops)
    return np.clip(out, 0, 255).astype(np.uint8)

show_variants(ORIGINAL, [
    ('stops = -2', aug_exposure(ORIGINAL, -2)),
    ('stops = -1', aug_exposure(ORIGINAL, -1)),
    ('stops = +1', aug_exposure(ORIGINAL, +1)),
    ('stops = +2', aug_exposure(ORIGINAL, +2)),
], 'Exposure simulation')

---
## 1.5 Fog / haze overlay

Blend the image with white (255) - simulates uniform haze or overexposed ambient light.

In [ ]:
def aug_fog(img: np.ndarray, strength: float) -> np.ndarray:
    """Blend image with white. strength=0 -> no effect, strength=1 -> pure white."""
    out = img.astype(np.float32) * (1 - strength) + 255.0 * strength
    return np.clip(out, 0, 255).astype(np.uint8)

show_variants(ORIGINAL, [
    ('strength = 0.1', aug_fog(ORIGINAL, 0.1)),
    ('strength = 0.3', aug_fog(ORIGINAL, 0.3)),
    ('strength = 0.5', aug_fog(ORIGINAL, 0.5)),
    ('strength = 0.7', aug_fog(ORIGINAL, 0.7)),
], 'Fog / haze overlay')

<a id="cat-noise"></a>
# Category 2: Noise

Augmentations that add pixel-level random noise to the image.

---
## 2.1 Gaussian noise

Add zero-mean Gaussian noise with standard deviation sigma.

In [ ]:
def aug_gaussian_noise(img: np.ndarray, sigma: float, seed: int = 0) -> np.ndarray:
    """Add Gaussian noise with std=sigma."""
    rng = np.random.default_rng(seed)
    noise = rng.normal(0, sigma, img.shape)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

show_variants(ORIGINAL, [
    ('sigma = 10',  aug_gaussian_noise(ORIGINAL, 10)),
    ('sigma = 25',  aug_gaussian_noise(ORIGINAL, 25)),
    ('sigma = 50',  aug_gaussian_noise(ORIGINAL, 50)),
    ('sigma = 80',  aug_gaussian_noise(ORIGINAL, 80)),
], 'Gaussian noise')

---
## 2.2 Salt-and-pepper noise

Randomly set a fraction of pixels to 0 (pepper) or 255 (salt).

In [ ]:
def aug_salt_pepper(img: np.ndarray, density: float, seed: int = 0) -> np.ndarray:
    """Replace `density` fraction of pixels with 0 or 255 (50/50 split)."""
    rng = np.random.default_rng(seed)
    out = img.copy()
    mask = rng.random(img.shape) < density
    salt = rng.random(img.shape) < 0.5
    out[mask & salt]  = 255
    out[mask & ~salt] = 0
    return out

show_variants(ORIGINAL, [
    ('density = 0.02', aug_salt_pepper(ORIGINAL, 0.02)),
    ('density = 0.05', aug_salt_pepper(ORIGINAL, 0.05)),
    ('density = 0.10', aug_salt_pepper(ORIGINAL, 0.10)),
    ('density = 0.20', aug_salt_pepper(ORIGINAL, 0.20)),
], 'Salt-and-pepper noise')

<a id="cat-lines"></a>
# Category 3: Random Lines

Random horizontal or vertical stripes with variable width and brightness - simulates sensor artifacts or physical scratches.

---
## 3.1 Random horizontal lines

N randomly placed horizontal stripes with random width (1-3 px) and random brightness.

In [ ]:
def aug_lines_h(
    img: np.ndarray,
    n_lines: int = 5,
    width_range: tuple = (1, 3),
    brightness_range: tuple = (0, 255),
    seed: int = 0,
) -> np.ndarray:
    """Draw N random horizontal lines with random width and brightness."""
    rng = np.random.default_rng(seed)
    out = img.copy()
    h = img.shape[0]
    for _ in range(n_lines):
        y      = rng.integers(0, h)
        width  = rng.integers(width_range[0], width_range[1] + 1)
        bright = rng.integers(brightness_range[0], brightness_range[1] + 1)
        y0 = max(0, y - width // 2)
        y1 = min(h, y0 + width)
        out[y0:y1, :] = bright
    return out

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, ax in enumerate(axes[0]):
    ax.imshow(aug_lines_h(ORIGINAL, n_lines=3, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=3  seed={i}', fontsize=9)
    ax.axis('off')
for i, ax in enumerate(axes[1]):
    ax.imshow(aug_lines_h(ORIGINAL, n_lines=8, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=8  seed={i}', fontsize=9)
    ax.axis('off')
fig.suptitle('Random horizontal lines', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3.2 Random vertical lines

Same as above but drawn column-wise.

In [ ]:
def aug_lines_v(
    img: np.ndarray,
    n_lines: int = 5,
    width_range: tuple = (1, 3),
    brightness_range: tuple = (0, 255),
    seed: int = 0,
) -> np.ndarray:
    """Draw N random vertical lines with random width and brightness."""
    rng = np.random.default_rng(seed)
    out = img.copy()
    w = img.shape[1]
    for _ in range(n_lines):
        x      = rng.integers(0, w)
        width  = rng.integers(width_range[0], width_range[1] + 1)
        bright = rng.integers(brightness_range[0], brightness_range[1] + 1)
        x0 = max(0, x - width // 2)
        x1 = min(w, x0 + width)
        out[:, x0:x1] = bright
    return out

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, ax in enumerate(axes[0]):
    ax.imshow(aug_lines_v(ORIGINAL, n_lines=3, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=3  seed={i}', fontsize=9)
    ax.axis('off')
for i, ax in enumerate(axes[1]):
    ax.imshow(aug_lines_v(ORIGINAL, n_lines=8, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=8  seed={i}', fontsize=9)
    ax.axis('off')
fig.suptitle('Random vertical lines', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id="cat-edge"></a>
# Category 4: Edge Effects

Directional fading and bilateral brightening/darkening along image edges.

---
## 4.1 Horizontal fading

Apply a linear ramp so one side of the image fades toward a target tone.

In [ ]:
def aug_fade_horizontal(
    img: np.ndarray,
    fade_to: int = 128,
    side: str = 'right',
    strength: float = 1.0,
) -> np.ndarray:
    """
    Fade one horizontal side toward fade_to grey value.
    side: 'left' or 'right'
    strength: 0=no effect, 1=full fade at edge
    """
    w = img.shape[1]
    ramp = np.linspace(0, strength, w, dtype=np.float32)
    if side == 'left':
        ramp = ramp[::-1]
    out = img.astype(np.float32) * (1 - ramp) + fade_to * ramp
    return np.clip(out, 0, 255).astype(np.uint8)

show_variants(ORIGINAL, [
    ('right->black', aug_fade_horizontal(ORIGINAL, fade_to=0,   side='right')),
    ('right->white', aug_fade_horizontal(ORIGINAL, fade_to=255, side='right')),
    ('left->black',  aug_fade_horizontal(ORIGINAL, fade_to=0,   side='left')),
    ('left->white',  aug_fade_horizontal(ORIGINAL, fade_to=255, side='left')),
], 'Horizontal fading')

---
## 4.2 Vertical fading

Same as horizontal fading but applied row-wise (top / bottom).

In [ ]:
def aug_fade_vertical(
    img: np.ndarray,
    fade_to: int = 128,
    side: str = 'bottom',
    strength: float = 1.0,
) -> np.ndarray:
    """
    Fade one vertical side toward fade_to grey value.
    side: 'top' or 'bottom'
    """
    h = img.shape[0]
    ramp = np.linspace(0, strength, h, dtype=np.float32).reshape(-1, 1)
    if side == 'top':
        ramp = ramp[::-1]
    out = img.astype(np.float32) * (1 - ramp) + fade_to * ramp
    return np.clip(out, 0, 255).astype(np.uint8)

show_variants(ORIGINAL, [
    ('bottom->black', aug_fade_vertical(ORIGINAL, fade_to=0,   side='bottom')),
    ('bottom->white', aug_fade_vertical(ORIGINAL, fade_to=255, side='bottom')),
    ('top->black',    aug_fade_vertical(ORIGINAL, fade_to=0,   side='top')),
    ('top->white',    aug_fade_vertical(ORIGINAL, fade_to=255, side='top')),
], 'Vertical fading')

---
## 4.3 Horizontal bilateral brightening

Both left and right edges fade toward a brighter/darker tone with Gaussian falloff - centre stays original.

In [ ]:
def aug_brighten_edges_h(
    img: np.ndarray,
    fade_to: int = 255,
    strength: float = 1.0,
    edge_fraction: float = 0.15,
) -> np.ndarray:
    """
    Brighten/darken both left and right edges with a Gaussian falloff.
    fade_to:       target pixel value at the edges (0=black, 255=white)
    strength:      peak blend weight at the outermost pixel
    edge_fraction: Gaussian sigma as fraction of half-width
    """
    w = img.shape[1]
    x = np.arange(w, dtype=np.float32)
    dist_from_edge = np.minimum(x, w - 1 - x)
    sigma = edge_fraction * (w / 2.0)
    weight = strength * np.exp(-(dist_from_edge ** 2) / (2 * sigma ** 2))
    out = img.astype(np.float32) * (1 - weight) + fade_to * weight
    return np.clip(out, 0, 255).astype(np.uint8)

show_variants(ORIGINAL, [
    ('10% bright', aug_brighten_edges_h(ORIGINAL, fade_to=255, edge_fraction=0.10)),
    ('20% bright', aug_brighten_edges_h(ORIGINAL, fade_to=255, edge_fraction=0.20)),
    ('10% dark',   aug_brighten_edges_h(ORIGINAL, fade_to=0,   edge_fraction=0.10)),
    ('20% dark',   aug_brighten_edges_h(ORIGINAL, fade_to=0,   edge_fraction=0.20)),
], 'Horizontal bilateral brightening / darkening (Gaussian)')

---
## 4.4 Vertical bilateral brightening

Both top and bottom edges fade toward a brighter/darker tone with Gaussian falloff - centre stays original.

In [ ]:
def aug_brighten_edges_v(
    img: np.ndarray,
    fade_to: int = 255,
    strength: float = 1.0,
    edge_fraction: float = 0.15,
) -> np.ndarray:
    """
    Brighten/darken both top and bottom edges with a Gaussian falloff.
    fade_to:       target pixel value at the edges (0=black, 255=white)
    strength:      peak blend weight at the outermost pixel
    edge_fraction: Gaussian sigma as fraction of half-height
    """
    h = img.shape[0]
    y = np.arange(h, dtype=np.float32)
    dist_from_edge = np.minimum(y, h - 1 - y)
    sigma = edge_fraction * (h / 2.0)
    weight = strength * np.exp(-(dist_from_edge ** 2) / (2 * sigma ** 2))
    weight = weight.reshape(-1, 1)
    out = img.astype(np.float32) * (1 - weight) + fade_to * weight
    return np.clip(out, 0, 255).astype(np.uint8)

show_variants(ORIGINAL, [
    ('10% bright', aug_brighten_edges_v(ORIGINAL, fade_to=255, edge_fraction=0.10)),
    ('20% bright', aug_brighten_edges_v(ORIGINAL, fade_to=255, edge_fraction=0.20)),
    ('10% dark',   aug_brighten_edges_v(ORIGINAL, fade_to=0,   edge_fraction=0.10)),
    ('20% dark',   aug_brighten_edges_v(ORIGINAL, fade_to=0,   edge_fraction=0.20)),
], 'Vertical bilateral brightening / darkening (Gaussian)')

<a id="cat-blur"></a>
# Category 5: Blur

Spatial smoothing applied uniformly across the image.

---
## 5.1 Gaussian blur

Smooth the image using a Gaussian kernel - simulates defocus or motion at a very fine scale.

In [ ]:
def aug_blur(img: np.ndarray, radius: float) -> np.ndarray:
    """Apply Gaussian blur with given radius (pixels)."""
    pil = Image.fromarray(img, mode='L')
    blurred = pil.filter(ImageFilter.GaussianBlur(radius=radius))
    return np.array(blurred)

show_variants(ORIGINAL, [
    ('radius = 1',  aug_blur(ORIGINAL, 1)),
    ('radius = 3',  aug_blur(ORIGINAL, 3)),
    ('radius = 6',  aug_blur(ORIGINAL, 6)),
    ('radius = 10', aug_blur(ORIGINAL, 10)),
], 'Gaussian blur')

<a id="cat-random"></a>
# Category 6: Random Profiles

Smooth random brightness waves along the horizontal or vertical axis,
built from a sum of Gaussians at random positions.

---
## 6.1 Random brightness profile - horizontal

A random smooth brightness wave applied column-wise. Built from N Gaussians at random
positions with random amplitudes - produces a complex but natural-looking variation.

In [ ]:
def aug_random_profile_h(
    img: np.ndarray,
    n_changes: int = 5,
    max_delta: float = 60.0,
    seed: int = 0,
) -> np.ndarray:
    """
    Apply a random smooth brightness profile along the horizontal axis.
    n_changes: number of Gaussian bumps along the width
    max_delta: maximum brightness shift (+/-) at any bump centre
    """
    w = img.shape[1]
    rng = np.random.default_rng(seed)
    sigma = w / (n_changes * 1.5)
    positions  = rng.uniform(0, w, n_changes)
    amplitudes = rng.uniform(-max_delta, max_delta, n_changes)
    x = np.arange(w, dtype=np.float32)
    profile = np.zeros(w, dtype=np.float32)
    for pos, amp in zip(positions, amplitudes):
        profile += amp * np.exp(-((x - pos) ** 2) / (2 * sigma ** 2))
    return np.clip(img.astype(np.float32) + profile, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, ax in enumerate(axes[0]):
    ax.imshow(aug_random_profile_h(ORIGINAL, n_changes=3, max_delta=60, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=3  seed={i}', fontsize=9)
    ax.axis('off')
for i, ax in enumerate(axes[1]):
    ax.imshow(aug_random_profile_h(ORIGINAL, n_changes=7, max_delta=60, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=7  seed={i}', fontsize=9)
    ax.axis('off')
fig.suptitle('Random brightness profile - horizontal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6.2 Random brightness profile - vertical

Same idea applied row-wise (top to bottom).

In [ ]:
def aug_random_profile_v(
    img: np.ndarray,
    n_changes: int = 5,
    max_delta: float = 60.0,
    seed: int = 0,
) -> np.ndarray:
    """
    Apply a random smooth brightness profile along the vertical axis.
    n_changes: number of Gaussian bumps along the height
    max_delta: maximum brightness shift (+/-) at any bump centre
    """
    h = img.shape[0]
    rng = np.random.default_rng(seed)
    sigma = h / (n_changes * 1.5)
    positions  = rng.uniform(0, h, n_changes)
    amplitudes = rng.uniform(-max_delta, max_delta, n_changes)
    y = np.arange(h, dtype=np.float32)
    profile = np.zeros(h, dtype=np.float32)
    for pos, amp in zip(positions, amplitudes):
        profile += amp * np.exp(-((y - pos) ** 2) / (2 * sigma ** 2))
    return np.clip(img.astype(np.float32) + profile.reshape(-1, 1), 0, 255).astype(np.uint8)

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, ax in enumerate(axes[0]):
    ax.imshow(aug_random_profile_v(ORIGINAL, n_changes=3, max_delta=60, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=3  seed={i}', fontsize=9)
    ax.axis('off')
for i, ax in enumerate(axes[1]):
    ax.imshow(aug_random_profile_v(ORIGINAL, n_changes=7, max_delta=60, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=7  seed={i}', fontsize=9)
    ax.axis('off')
fig.suptitle('Random brightness profile - vertical', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id="cat-combo"></a>
# Combination - chained augmentations

Apply augmentations from different categories in sequence to see cumulative effect.

In [ ]:
steps = [
    ('original',             ORIGINAL),
    ('+brightness -30',      aug_brightness(ORIGINAL, -30)),
    ('+contrast x1.4',       aug_contrast(aug_brightness(ORIGINAL, -30), 1.4)),
    ('+gaussian noise s=15', aug_gaussian_noise(aug_contrast(aug_brightness(ORIGINAL, -30), 1.4), 15)),
    ('+fog 0.2',             aug_fog(aug_gaussian_noise(aug_contrast(aug_brightness(ORIGINAL, -30), 1.4), 15), 0.2)),
    ('+blur r=1',            aug_blur(aug_fog(aug_gaussian_noise(aug_contrast(aug_brightness(ORIGINAL, -30), 1.4), 15), 0.2), 1)),
]

fig, axes = plt.subplots(1, len(steps), figsize=(len(steps) * 3, 3))
for ax, (label, img) in zip(axes, steps):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    ax.set_title(label, fontsize=8)
    ax.axis('off')
fig.suptitle('Chained augmentations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()